# This notebook makes quick previews of SWOT Unsmoothed data (L2)

In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from ipywidgets import IntSlider, Dropdown, HBox, VBox, Output
from IPython.display import display, clear_output
import cmocean  # Import cmocean for colormaps
import cmocean.cm as cmo
import os
import sys as sys
sys.path.append('../src/')
from  spectral_analysis_functions import *

lightcmap = cmocean.tools.lighten(cmo.ice, 1)

# Your cross-track and along-track resolutions
dx = 250  # cross-track resolution in meters
dy = 235  # along-track resolution in meters
#nj=60;njf=42;
nj=120;njf=84
#ni=120;i1=40;nif=40;
i1=0;nif=80;ni=240
thetai=785E3

# Folder containing the SWOT files
folder_path = '/media/ardhuin/FabBack/SWOT2/Unsmoothed/'

# File path
j0 = 67000
ssh_file = 'SWOT_L2_LR_SSH_Unsmoothed_567_028_20230630T063018_20230630T072124_PGD0_01.nc'
ssh_file = 'SWOT_L2_LR_SSH_Unsmoothed_570_028_20230703T060212_20230703T065318_PGD0_01.nc'
h=857000

#j0 = 19480
#ssh_file = 'SWOT_L2_LR_SSH_Unsmoothed_009_024_20240104T231743_20240105T000910_PGD0_02.nc'
#h=891000

#h=857000
#j0 = 19480
#ssh_file = '/media/ardhuin/FabBack/SWOT2/Unsmoothed/SWOT_L2_LR_SSH_Unsmoothed_480_005_20230404T003014_20230404T012120_PGD0_01.nc'



# List all NetCDF files in the folder
file_list = [f for f in os.listdir(folder_path) if f.endswith('.nc')]
file_list.sort()  # Sort alphabetically

# Create a dropdown menu for file selection
file_dropdown = Dropdown(
    options=file_list,
    value=ssh_file,  # Default: first file in the list
    description='SWOT File:',
    disabled=False,
)


In [2]:
# A few useful numbers: orders of magnitude of kz for SWOT 
k0B=785*10

Hss=0.1
Ess=(Hss/4)**2
EssPSD=Ess/9/(0.0002**2)
print(Ess,EssPSD,10*np.log10(EssPSD))
print(2*np.pi/(8-2*np.pi))


0.0006250000000000001 1736.1111111111113 32.39577516576788
3.659792366325487


In [3]:

# Shared along-track index slider
along_slider = IntSlider(
    min=120, max=82200, step=80,
    value=j0, description="along-track:"
)

# Tile size slider

box_slider = IntSlider(
    min=20, max=240, step=10,
    value=160, description="FFT box size"
)


tile_slider = IntSlider(
    min=1, max=8, step=1,
    value=8, description="number of FFT tiles"
)

# Side and variable selection for Panel 1
side_slider1 = IntSlider(
    min=0, max=1, step=1,
    value=0, description="side 1:"
)
var_dropdown1 = Dropdown(
    options=["SSH", "SSHA", "sigma0", "SWH", "vol. decor.", "uncertainty"],
    value="SSH",
    description="Variable 1:",
)

# Side and variable selection for Panel 2
side_slider2 = IntSlider(
    min=0, max=1, step=1,
    value=0, description="side 2:"
)
var_dropdown2 = Dropdown(
    options=["SSH", "SSHA", "sigma0", "SWH","vol. decor.", "uncertainty"],
    value="sigma0",
    description="Variable 2:",
)

var_dropdown3 = Dropdown(
    options=["None", "PSD 1", "PSD 2", "coh. 2-1", "phase 2-1"],
    description="Spectrum:",
)


# Mapping from variable names to NetCDF variable names, labels, and colormaps
var_mapping = {
    "SSH": {"L2_variable": "ssh_karin_2", "label": "ssh (m)", "cmap": lightcmap},  # Your custom lightcmap for SSH
    "SSHA": {"L2_variable": "ssha_karin_2", "label": "ssh (m)", "cmap": lightcmap},  # Your custom lightcmap for SSH
    "sigma0": {"L2_variable": "sig0_karin_2", "label": "sigma0 (linear)", "cmap": "Greys_r"},
    "SWH": {"L2_variable": "volumetric_correlation", "label": "SWH (m)", "cmap": "viridis"},  
    "vol. decor.": {"L2_variable": "volumetric_correlation", "label": "vol. decor.", "cmap": "viridis"},
    "uncertainty": {"L2_variable": "ssh_karin_uncert", "label": "uncertainty (cm)", "cmap": "viridis"},
}

# Output widget to display the plot
out = Output()

# Function to update the plot
def update_plot(change):
    with out:
        clear_output(wait=True)
        j = along_slider.value
        s1 = side_slider1.value
        var1 = var_dropdown1.value
        s2 = side_slider2.value
        var2 = var_dropdown2.value
        ntile=tile_slider.value
        spec =var_dropdown3.value
        ssh_file=file_dropdown.value

        print('file:',ssh_file)
# selection of data along track 
        j1 = j - nj
        j2 = j + nj
        i2 = i1 + ni
        print('J index:',j1,j2,i1,i2)

        fig = plt.figure(figsize=(18, 8))
        gs = gridspec.GridSpec(
            3, 3,
            height_ratios=[12,18, 2],  # Small row for colorbars
            width_ratios=[1, 1, 0.5], # First two panels: equal width, third panel: narrower
            hspace=0.35,
            wspace=0.12,
        )

        # Main panels
        axs = [
            fig.add_subplot(gs[0:2, 0]),  # Panel 1
            fig.add_subplot(gs[0:2, 1]),  # Panel 2
            fig.add_subplot(gs[1, 2]),  # Reserved for spectra/cross-spectra
        ]

        # Colorbar panels
        cax0 = fig.add_subplot(gs[2, 0])
        cax1 = fig.add_subplot(gs[2, 1])
        cax2 = fig.add_subplot(gs[2, 2])  # Unused for now

        fs1 = 20

        # ---  Variable  1 ---
        ddl1 = xr.open_dataset(folder_path+ssh_file, group='left' if s1 == 0 else 'right')
        var_info1 = var_mapping[var1]
        L2_variable1 = var_info1["L2_variable"]
        label1 = var_info1["label"]
        cmap1 = var_info1["cmap"]

        # Slice the dataset first (lazy loading)
        lat1 = ddl1.latitude[j1:j2, i1:i2].values
        lon1 = ddl1.longitude[j1:j2, i1:i2].values
        if s1 == 0:
            dataSWOT1 = np.flip(ddl1[L2_variable1][j1:j2, i1:i2].values, axis=1)  # Slice, then flip, then convert to NumPy
            ix1 = -240 - 18 + i1
        else:
            dataSWOT1 = ddl1[L2_variable1][j1:j2, i1:i2].values  # Slice, then convert to NumPy
            ix1 = i1 + 19

        # --- variable 2

        ddl2 = xr.open_dataset(folder_path+ssh_file, group='left' if s2 == 0 else 'right')
        var_info2 = var_mapping[var2]
        L2_variable2 = var_info2["L2_variable"]
        label2 = var_info2["label"]
        cmap2 = var_info2["cmap"]

        # Slice the dataset first (lazy loading)
        lat2 = ddl2.latitude[j1:j2, i1:i2].values
        lon2 = ddl2.longitude[j1:j2, i1:i2].values
        if s2 == 0:
            dataSWOT2 = np.flip(ddl2[L2_variable2][j1:j2, i1:i2].values, axis=1)  # Slice, then flip, then convert to NumPy
            ix2 = -240 - 18 + i1
        else:
            dataSWOT2 = ddl2[L2_variable2][j1:j2, i1:i2].values  # Slice, then convert to NumPy
            ix2 = i1 + 19
    

        detrend1=dataSWOT1
        detrend2=dataSWOT2

        if var1=="SWH":
            if s1 == 0:
                Xd=np.flip(ddl1["cross_track_distance"][j1:j2, i1:i2].values, axis=1) 
            else:
                Xd=ddl1["cross_track_distance"][j1:j2, i1:i2].values
            dist=np.sqrt(Xd**2+h**2)
            sinth=np.abs(Xd)/dist
            kz=k0B/(h*sinth)
            detrend1=np.sqrt(np.abs(-16*np.log(dataSWOT1)))/kz
            gmed=np.nanmedian(dataSWOT1.flatten())
            print('SWH median:',np.nanmedian(detrend1.flatten()),gmed) 

        if var2=="SWH":
            if s1 == 0:
                Xd=np.flip(ddl1["cross_track_distance"][j1:j2, i1:i2].values, axis=1) 
            else:
                Xd=ddl1["cross_track_distance"][j1:j2, i1:i2].values
            dist=np.sqrt(Xd**2+h**2)
            sinth=np.abs(Xd)/dist
            kz=k0B/(h*sinth)
            detrend2=np.sqrt(np.abs(-16*np.log(dataSWOT2)))/kz
            gmed=np.nanmedian(dataSWOT2.flatten())
            print('SWH median:',np.nanmedian(detrend2.flatten()),gmed) 


        if (spec=='None') & (var1=="sigma0"):
            detrend1=10*np.log10(detrend1)
            label1='sigma0 (dB)'
        if (spec=='None') & (var2=="sigma0"):
            detrend2=10*np.log10(detrend2)
            label2='sigma0 (dB)'

        amax1 = np.nanmax(detrend1)
        amin1 = np.nanmin(detrend1)

        amax2 = np.nanmax(detrend2)
        amin2 = np.nanmin(detrend2)

        anomaly=''
        if spec!='None':
            anomaly=' anomaly'
            temp=detrend1.copy()
            detrend1=detrend_2d_quadratic_nan(temp)
            temp=detrend2.copy()
            detrend2=detrend_2d_quadratic_nan(temp)
            ic=(i1+i2)//2
            jc=(j1+j2)//2-j1
            if L2_variable1=="sig0__karin_2":
               myvar1=10**(0.1*dataSWOT2[jc-njf:jc+njf,ic-nif:ic+nif])
            else:
               myvar1=dataSWOT1[jc-njf:jc+njf,ic-nif:ic+nif]

            if L2_variable2=="sig0__karin_2":
               print('TEST: sig0')  
               myvar2=10**(0.1*dataSWOT2[jc-njf:jc+njf,ic-nif:ic+nif])
            else:
               print('TEST: NOT sig0')  
               myvar2=detrend2[jc-njf:jc+njf,ic-nif:ic+nif]
            print('size:',njf,'##',ic,nif,ic-nif,ic+nif,np.shape(myvar1),'##',np.shape(dataSWOT1))
            (PSD1,PSD2,ang,angstd,coh,crosr,phases,ky2,kx2,dky,dkx,detrend1b,detrend2b,nspec)=FFT2D_two_arrays_nm_detrend_flag(myvar1,myvar2,myvar1*0+1, \
                                                                                                     dy,dx,ntile,ntile,detrend='quadratic') 

            amax1 = np.nanmax(detrend1b)
            amin1 = np.nanmin(detrend1b)
            detrend1[jc-njf:jc+njf,ic-nif:ic+nif]=detrend1b
            if L2_variable2=="sig0__karin_2":
               detrend2[jc-njf:jc+njf,ic-nif:ic+nif]=10*np.log10(detrend2b)
            else:
               detrend2[jc-njf:jc+njf,ic-nif:ic+nif]=detrend2b
            amax2 = np.nanmax(detrend2[jc-njf:jc+njf,ic-nif:ic+nif])
            amin2 = np.nanmin(detrend2[jc-njf:jc+njf,ic-nif:ic+nif])

            
        # --- Panel 1 (same region, possibly different side and variable) ---
        latmean1 = 0.5 * (lat1[0,20] + lat1[0, 20])
        lonmean1 = 0.5 * (lon1[-1,-20] + lon1[-1,-20])

        X1 = (np.arange(dataSWOT1.shape[1]) + ix1+1) * dx / 1000
        Y1 = (np.arange(dataSWOT1.shape[0]) - j1) * dy / 1000
        Y1 = Y1 - Y1[0]

        im0 = axs[0].pcolormesh(X1, Y1, detrend1, cmap=cmap1, vmin=amin1, vmax=amax1, rasterized=True)
        fig.colorbar(im0, cax=cax0, orientation='horizontal', label=label1+anomaly)
        axs[0].set_xlabel('cross-track (km)', fontsize=fs1)
        axs[0].set_ylabel('along-track (km)', fontsize=fs1)
        axs[0].set_title(f'lon={lonmean1:.2f}°, lat={latmean1:.2f}° (Side: {"left" if s1 == 0 else "right"})')


        # --- Panel 2 (same region, possibly different side and variable) ---
        latmean2 = 0.5 * (lat2[0,20] + lat2[0, 20])
        lonmean2 = 0.5 * (lon2[-1,-20] + lon2[-1,-20])
        X2 = (np.arange(dataSWOT2.shape[1]) + ix2) * dx / 1000

        im1 = axs[1].pcolormesh(X2, Y1, detrend2, cmap=cmap2, vmin=amin2, vmax=amax2, rasterized=True)
        fig.colorbar(im1, cax=cax1, orientation='horizontal', label=label2+anomaly)
        axs[1].set_xlabel('cross-track (km)', fontsize=fs1)
        axs[1].set_yticklabels([])  # Remove y-axis labels for Panel 2
        axs[1].set_title(f'lon={lonmean2:.2f}°, lat={latmean2:.2f}° (Side: {"left" if s2 == 0 else "right"})')

        # --- Panel 3 (Reserved for spectra/cross-spectra) ---
        if spec=="PSD 1":
            im2 = axs[2].pcolormesh(kx2 * 1000,ky2 * 1000,10 * np.log10(PSD1),norm=mcolors.Normalize(vmin=0, vmax=50),rasterized=True, cmap='viridis')
            #vertices=swell.SWOTspec_mask_polygon(bmask) 
            #swell.draw_mask(axs[1],kx2s,dkx,ky2s,dky,vertices,color='w',lw=3)
            axs[2].set_title("PSD of 1", fontsize=fs1)
            fig.colorbar(im2, cax=cax2, orientation='horizontal', label='')

        if spec=="PSD 2":
            im2 = axs[2].pcolormesh(kx2 * 1000,ky2 * 1000,10 * np.log10(PSD2),norm=mcolors.Normalize(vmin=0, vmax=50),rasterized=True, cmap='viridis')
            #vertices=swell.SWOTspec_mask_polygon(bmask) 
            #swell.draw_mask(axs[1],kx2s,dkx,ky2s,dky,vertices,color='w',lw=3)
            axs[2].set_title("PSD of 2", fontsize=fs1)
            #axs[2].axis('off')  # Hide axes for now
            fig.colorbar(im2, cax=cax2, orientation='horizontal', label='')

        if spec=="coh. 2-1":
            im2 = axs[2].pcolormesh(kx2 * 1000,ky2 * 1000,coh,norm=mcolors.Normalize(vmin=0, vmax=1),rasterized=True, cmap='viridis')
            #vertices=swell.SWOTspec_mask_polygon(bmask) 
            #swell.draw_mask(axs[1],kx2s,dkx,ky2s,dky,vertices,color='w',lw=3)
            axs[2].set_title("coherence(2,1) ", fontsize=fs1)
            fig.colorbar(im2, cax=cax2, orientation='horizontal', label='')

        if spec=="phase 2-1":
            print('test:',ntile,ang.shape[1],np.nanmax(ang))
            im2 = axs[2].pcolormesh(kx2 * 1000,ky2 * 1000,ang*(180/np.pi),norm=mcolors.Normalize(vmin=-180, vmax=180),rasterized=True, cmap='seismic')
            #vertices=swell.SWOTspec_mask_polygon(bmask) 
            #swell.draw_mask(axs[1],kx2s,dkx,ky2s,dky,vertices,color='w',lw=3)
            axs[2].set_title("phase 2-1", fontsize=fs1)
            #axs[2].axis('off')  # Hide axes for now
            cbar=fig.colorbar(im2, cax=cax2, orientation='horizontal', label='')
            # Tick positions
            cbar.set_ticks([-180, -90, 0, 90, 180]) 
        
        plt.show()

# Create a two-column layout for the widgets
controls = HBox([
    VBox([side_slider1, var_dropdown1]),  # Column 1: Side 1 and Variable 1
    VBox([side_slider2, var_dropdown2]),   # Column 2: Side 2 and Variable 2
    VBox([tile_slider, var_dropdown3])   # Column 2: Side 2 and Variable 2
])

# Display the widgets manually
display(along_slider)
display(file_dropdown)
#display(box_slider)
display(controls)
display(out)

# Register the update function with all widgets
for widget in [along_slider, tile_slider, side_slider1, var_dropdown1, side_slider2, var_dropdown2]:
    widget.observe(update_plot, names='value')

# Initial plot
update_plot(None)

IntSlider(value=67000, description='along-track:', max=82200, min=120, step=80)

Dropdown(description='SWOT File:', index=1252, options=('SWOT_L2_LR_SSH_Unsmoothed_001_485_20230807T123410_202…

Output()

In [17]:
from scipy.signal import detrend  # For linear detrending
import matplotlib as mpl
mpl.rcParams.update({'figure.figsize':[10,6],'axes.grid' : True,'font.size': 14,'savefig.facecolor':'white'})
from ipywidgets import interact, IntSlider, HBox, VBox
from scipy.signal import detrend, butter, filtfilt
from scipy.interpolate import interp1d

j0 = along_slider.value

j1_slider = IntSlider(min=0, max=240, step=1, value=150, description='j1:')  # New slider for j1

s1 = side_slider1.value
var1 = var_dropdown1.value
s2 = side_slider2.value
var2 = var_dropdown2.value
ssh_file=file_dropdown.value
i1D = 30;    ni1D = 180;    i2 = i1 + ni1D

# Butterworth low-pass filter function
def butter_lowpass(y, cutoff_freq=0.1, fs=1.0, order=3):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff_freq / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, y)

# Define the plotting function
def plot_transect(j1):
    j = j0-nj + j1
    Y1 = j1 * dy / 1000.
    X1 = (j1+19) * dx / 1000.
    print('Y:', Y1, X1, folder_path + ssh_file)


    ddl1 = xr.open_dataset(folder_path + ssh_file, group='left' if s1 == 0 else 'right')
    ddl2 = xr.open_dataset(folder_path + ssh_file, group='left' if s2 == 0 else 'right')
    var_info1 = var_mapping[var1]
    var_info2 = var_mapping[var2]
    L2_variable1 = var_info1["L2_variable"]
    L2_variable2 = var_info2["L2_variable"]
    label1 = var_info1["label"]
    label2 = var_info2["label"]

    if s1 == 0:
        y1 = ddl1[L2_variable1][j, i1D:i2].values
        y2 = ddl2[L2_variable2][j, i1D:i2].values
        ix1 = -240 - 18 + i1D
    else:
        y1 = ddl1[L2_variable1][j, i1D:i2].values
        y2 = ddl2[L2_variable2][j, i1D:i2].values
        y1 = ddl1[L2_variable1][j0-nj:j0+nj, j1].values
        y2 = ddl2[L2_variable2][j0-nj:j0+nj, j1].values
        ix1 = i1D + 19

    if var2 == "SWH":
        if s1 == 0:
            Xd = np.flip(ddl1["cross_track_distance"][j, i1D:i2].values, axis=1)
        else:
            Xd = ddl1["cross_track_distance"][j, i1D:i2].values
            Xd = ddl1["cross_track_distance"][j0-nj:j0+nj, j1].values
            dist = np.sqrt(Xd**2 + h**2)
            sinth = np.abs(Xd) / dist
            kz = k0B / (h * sinth)
            y2 = np.sqrt(np.abs(-16 * np.log(y2))) / kz

    X1 = (np.arange(len(y1)) + ix1+1) * dx / 1000
    X1 = (np.arange(len(y1)) ) * dy / 1000

    # Detrend y1 and y2
    y1_detrended = detrend(y1, type='linear')
    y2_detrended = detrend(y2, type='linear')
    # Apply low-pass filter to y1_detrended
    y1_filtered = butter_lowpass(y1_detrended, cutoff_freq=0.07, fs=1.0, order=3)
    mask = np.isfinite(y1)
    Xd_clean = Xd[mask]/1000.
    y1_clean = y1[mask]
    # Interpolate only valid points onto X1
    #y1_interp = detrend(np.interp(X1[5:-5], Xd_clean, y1_clean, left=np.nan, right=np.nan), type='linear')

    # Restore NaNs where y1 was originally NaN
    #y1_interp[~mask] = np.nan
    
    #y1_interp_filtered = butter_lowpass(y1_interp, cutoff_freq=0.07, fs=1.0, order=3) 
    # Plot
    plt.figure(figsize=(10, 4))
    plt.plot(X1, 0.02 * detrend(Xd - X1 * 1000, type='linear'), 'k-', lw=1, label=r'$0.02 (x-x_r)$ (m) (detrended)')
    plt.plot(X1, y1_detrended, 'b-', lw=1, label=f"{label1} (detrended)")
    plt.plot(X1[10:-10], y1_filtered[10:-10]*5, 'g-', lw=2, label=f"{label1} (5x low-pass filtered)")
    #plt.plot(X1[10:-10], y1_interp_filtered[5:-5]*5, 'm-', lw=2, label=f"{label1} (low-pass filtered x5: z(r) )")
    plt.plot(X1, y2_detrended, 'r-', lw=2, label=f"{label2} (detrended)")
    plt.xlabel("x (km)")
    plt.ylabel("Detrended SSH or SWH")
    plt.legend()
    plt.grid(True)
    plt.savefig('transect.pdf')
    plt.show()

# Organize the widgets
widgets = VBox([
    along_slider,
    j1_slider,  # Add the j1 slider
    HBox([side_slider1, var_dropdown1]),
    HBox([side_slider2, var_dropdown2]),
    file_dropdown
])

# Set up the interactive plot
interact(
    plot_transect,
    j1=j1_slider)




interactive(children=(IntSlider(value=150, description='j1:', max=240), Output()), _dom_classes=('widget-inter…

<function __main__.plot_transect(j1)>

In [123]:
np.log(27/16)/0.46

1.1374959647055387

In [130]:
10*np.log10(16*np.exp(0.46*0))

12.041199826559248